# V11: lower-rate continuation from V10 (P=10)
This experiment keeps V10's 40-layer pilot-conditioned architecture, training data, loss, and validation checkpoint rule fixed. It continues the selected V10 models for up to 100 more epochs at a lower learning rate. The goal is to test whether additional optimization alone closes the high-SNR validation gap. The test split is not loaded.

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')
import os, torch
root='/content/gdrive/MyDrive'
required=[f'{root}/DL_dataset_RSR25dB/{s}_P10.mat' for s in ('train','validation')]
required += [f'{root}/DL_dataset_RSR25dB/v6_starting_models/gs_validation_P10.json']
required += [f'{root}/unfolded_dl_v10_pilot_conditioned_p10_results/model_P10_seed{seed}_v10.pt' for seed in (20261026,20261027,20261028)]
missing=[p for p in required if not os.path.isfile(p)]
assert not missing, f'Missing Drive files: {missing}'
print('P=10 train/validation data, GS validation reference, and all three V10 checkpoints are present.')
print('Runtime:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

## Upload the scripts
Upload all eight files from your workspace `v2` folder: `colab_unfolded_dl_v3_highsnr.py`, `colab_unfolded_dl_v5_snr_aware.py`, `colab_unfolded_dl_v6_gs_gap.py`, `colab_unfolded_dl_v7_continue_p10.py`, `colab_unfolded_dl_v8_continue_p10.py`, `colab_unfolded_dl_v9_highsnr_finetune.py`, `colab_unfolded_dl_v10_pilot_conditioned_p10.py`, and `colab_unfolded_dl_v11_continue_v10_p10.py`.

In [ ]:
from google.colab import files
uploaded=files.upload()
required_scripts={
 'colab_unfolded_dl_v3_highsnr.py','colab_unfolded_dl_v5_snr_aware.py',
 'colab_unfolded_dl_v6_gs_gap.py','colab_unfolded_dl_v7_continue_p10.py',
 'colab_unfolded_dl_v8_continue_p10.py','colab_unfolded_dl_v9_highsnr_finetune.py',
 'colab_unfolded_dl_v10_pilot_conditioned_p10.py',
 'colab_unfolded_dl_v11_continue_v10_p10.py'}
missing=required_scripts.difference(uploaded)
assert not missing, f'Upload these scripts: {sorted(missing)}'
print('All eight scripts uploaded.')

## Run the continuation
This may take a while on CPU. The script saves resumable checkpoints and selected models to a new Drive folder. It uses the same 4? weights at 25 and 30 dB as V9/V10 and picks checkpoints by the worst validation DL?GS gap across all eight SNRs. No test data is opened.

In [ ]:
!python /content/colab_unfolded_dl_v11_continue_v10_p10.py

## Compare selected validation gaps
Negative DL?GS favors DL. This cell reads only V10 and V11 validation summaries.

In [ ]:
import json, numpy as np
root='/content/gdrive/MyDrive'
p10=f'{root}/unfolded_dl_v10_pilot_conditioned_p10_results/summary_unfolded_v10_pilot_condition_p10.json'
p11=f'{root}/unfolded_dl_v11_continue_v10_p10_results/summary_unfolded_v11_continue_v10_p10.json'
with open(p10,encoding='utf-8') as f: v10=json.load(f)
with open(p11,encoding='utf-8') as f: v11=json.load(f)
snrs=[-5,0,5,10,15,20,25,30]
print('SNR | V10 mean DL-GS | V11 mean DL-GS | V11 seeds beating GS')
for snr in snrs:
 key=str(snr)
 a=[r['selected_validation_dl_minus_gs_db_by_snr'][key] for r in v10]
 b=[r['selected_validation_dl_minus_gs_db_by_snr'][key] for r in v11]
 print(f'{snr:>3} dB | {np.mean(a):+8.4f} dB | {np.mean(b):+8.4f} dB | {sum(x<0 for x in b)}/3')
print('V11 validation outputs:',f'{root}/unfolded_dl_v11_continue_v10_p10_results')